# python-control — control system design

**What this library does in one sentence:** `control` is a Python port of MATLAB's Control Systems Toolbox — state-space models, linear quadratic regulators, transfer functions, pole placement, frequency-domain plots.

For this project, `control` is how we *design* the controller that balances the inverted pendulum. SciPy simulates the dynamics; `control` decides what the cart should push to keep the pole upright.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import control

## 1. What control theory is — in one paragraph

**Control theory** is about *closing the loop*: measure the system's state, compute a corrective input, apply it, repeat. For the cart-pole, the loop reads the pole angle `θ`, decides how hard to push the cart, applies that force, and runs again 50 times a second. The hard part is choosing *how* to map (state) → (force) so the pole doesn't fall over. That mapping is the **controller**, and designing it is what this library is for.

## 2. State-space representation

A linear system can always be written as

$$\dot{x} = A x + B u, \qquad y = C x + D u$$

where

- **x** is the state vector (for cart-pole: `[x, ẋ, θ, θ̇]`)
- **u** is the input vector (for cart-pole: scalar force `F`)
- **y** is the output vector — whatever you can measure

The four matrices have specific physical meaning:

| Matrix | Shape         | What it encodes |
|--------|---------------|-----------------|
| **A**  | `(n, n)`      | Internal dynamics — how state evolves on its own. |
| **B**  | `(n, m)`      | Input coupling — how the input pushes each state. |
| **C**  | `(p, n)`      | Output map — which states (or combinations) you observe. |
| **D**  | `(p, m)`      | Direct feedthrough — input that bypasses dynamics. Usually zero. |

For the linearised cart-pole near the upright equilibrium (we derive this later in this notebook):

```
A = 4x4 matrix containing  g, M, m, L
B = 4x1 column            containing 1/M and similar
C = 4x4 identity          (we measure all four states)
D = 4x1 zeros
```

### `control.ss(A, B, C, D)`

Build a continuous-time state-space system object.

| Parameter | Meaning |
|-----------|---------|
| `A, B, C, D` | The four matrices above (NumPy arrays). |

Returns a `StateSpace` object. Its matrices are accessible as `sys.A`, `sys.B`, etc.

In [ ]:
# minimal — a simple double integrator (think: mass on a frictionless line)
A = np.array([[0, 1],
              [0, 0]])
B = np.array([[0],
              [1]])
C = np.eye(2)
D = np.zeros((2, 1))

sys = control.ss(A, B, C, D)
print(sys)

## 3. Linearization context — why we need it

The cart-pole EOM is **nonlinear** (it has `sin θ`, `cos θ`, products of states). `python-control` works with **linear** systems. So we **linearize** the dynamics around an operating point — for us, the upright equilibrium `θ = 0`.

Near the upright position `sin θ ≈ θ` and `cos θ ≈ 1`, the equations become linear in `(x, ẋ, θ, θ̇)`, fitting the `ẋ = Ax + Bu` form. Then `control.lqr` can design a controller that works **as long as the pole stays near vertical**.

If you push the pole 60° off vertical, the linearised model is wrong and the controller may fail. That's the price of linearisation — and a topic for nonlinear control (out of scope here).

## 4. LQR design

### What LQR is

**Linear Quadratic Regulator** is the optimal state-feedback controller that minimises

$$J = \int_0^\infty \left( x^T Q\, x + u^T R\, u \right) dt$$

In words: **"minimise a weighted sum of how far you are from the goal (Q) plus how much effort you used (R), summed over all time."** The result is a constant gain matrix `K` such that the control law `u = -K x` is optimal.

You don't need to solve that integral. `control.lqr` solves a matrix equation (the algebraic Riccati equation) and hands you `K`. You only need to supply `Q` and `R`.

### The Q and R matrices — intuition

- **Q** is `(n × n)`, symmetric. **Bigger Q means "care more about state error."** Usually diagonal, with the i-th entry weighting how much you penalise the i-th state being non-zero.
- **R** is `(m × m)`, symmetric. **Bigger R means "penalise control effort more."** Usually diagonal.

**Tuning heuristic:** start with `Q = I`, `R = 1`. If the response is too sluggish, increase the relevant diagonal entry of `Q`. If the controller asks for huge forces (motor saturates), increase `R`.

For cart-pole, you'll typically penalise `θ` much more than `x` — you care that the pole stays upright; the cart's exact position is less critical.

### `control.lqr(sys, Q, R)` — full signature

| Parameter | Meaning |
|-----------|---------|
| `sys` (or `A, B`) | The system, or the `A` and `B` matrices directly. |
| `Q`       | `(n, n)` state-error cost. |
| `R`       | `(m, m)` control-effort cost. |

**Returns** a tuple `(K, S, E)`:

| Return | Shape         | Meaning |
|--------|---------------|---------|
| `K`    | `(m, n)`      | The feedback gain — use as `u = -K @ x`. |
| `S`    | `(n, n)`      | Solution of the Riccati equation. Diagnostic; rarely used directly. |
| `E`    | `(n,)` complex| Eigenvalues of the closed-loop system. All should have negative real parts. |

In [ ]:
# minimal — LQR for the double integrator
Q = np.diag([10, 1])          # position error matters more than velocity error
R = np.array([[1.0]])         # modest penalty on control effort

K, S, E = control.lqr(sys, Q, R)
print('K =', K)
print('closed-loop eigenvalues:', E)

### The `K` matrix and the control law

`K` has shape `(m, n)` — `m` inputs by `n` states. To compute the control force from the current state:

```python
u = -K @ state            # the minus is part of the law, not the matrix
```

For a single input (`m=1`) and a 4-state system, `K` is a `(1, 4)` row vector. `K @ state` gives a scalar, then negated → scalar force `u`.

### Closed-loop system

Applying `u = -K x` to `ẋ = A x + B u` gives

$$\dot{x} = (A - B K) x$$

The **closed-loop A** matrix is `A_cl = A - B @ K`. The eigenvalues of `A_cl` are the closed-loop poles — they tell you whether the system is stable.

In [ ]:
A_cl = A - B @ K
print('A_cl =')
print(A_cl)
print('A_cl eigenvalues:', np.linalg.eigvals(A_cl))
# should match E returned by lqr

## 5. Analysis tools

### `control.poles(sys)` — stability at a glance

Returns the eigenvalues of `A`. **A linear system is stable iff all poles have negative real part.**

```python
control.poles(sys)
```

- All negative reals  →  stable, no oscillation
- Complex pair with negative real  →  stable, damped oscillation
- Any pole on or right of the imaginary axis  →  unstable

In [ ]:
print('open-loop poles  :', control.poles(sys))
# both at 0 — marginally stable. The closed-loop version should look much better.

sys_cl = control.ss(A - B @ K, B, C, D)
print('closed-loop poles:', control.poles(sys_cl))

### `control.step_response(sys)`

Simulate the system's response to a step input — "if I suddenly apply `u=1` at `t=0`, what does the state do?"

Returns `(t, y)`.

In [ ]:
t_resp, y_resp = control.step_response(sys_cl)
plt.plot(t_resp, y_resp[0], label='state 1 (position)')
plt.plot(t_resp, y_resp[1], label='state 2 (velocity)')
plt.xlabel('t'); plt.ylabel('y'); plt.legend(); plt.grid(alpha=0.3)
plt.title('closed-loop step response'); plt.show()

**Reading it:** if the curves *settle* to a finite value (not blow up, not oscillate forever), the controller is stable. Faster settling = more aggressive controller (typically bigger `Q` or smaller `R`).

### `control.bode_plot(sys)`

Frequency-domain analysis — "if I drive the input as `sin(ωt)`, what's the output amplitude and phase shift, as a function of `ω`?"

Less important for LQR design but useful for spotting resonances. Look for **peaks** (resonant frequencies) and **rolloff** (where the system stops responding to fast inputs).

```python
control.bode_plot(sys)
```

## 6. Stability check — the workflow

```python
poles_open  = control.poles(sys)
K, _, _     = control.lqr(sys, Q, R)
sys_cl      = control.ss(sys.A - sys.B @ K, sys.B, sys.C, sys.D)
poles_close = control.poles(sys_cl)

assert (np.real(poles_close) < 0).all(), "controller did not stabilise!"
print('open-loop  poles:', poles_open)
print('closed-loop poles:', poles_close)
```

For an unstable open-loop system (like the cart-pole at upright), `poles_open` will contain at least one pole with **positive** real part. After LQR, every pole should have **negative** real part — the controller has *pulled the poles into the left half-plane*.

## 7. Worked example — LQR for the linearised cart-pole

We linearize around the upright equilibrium `(x*, ẋ*, θ*, θ̇*) = (0, 0, 0, 0)` with `F* = 0`. Using `sin θ ≈ θ`, `cos θ ≈ 1`, and dropping all `(θ)(θ̇)`-type product terms, the EOM becomes:

```
ẍ  ≈  (-m g θ + F) / M
θ̈  ≈  ((M+m) g θ - F) / (M L)
```

With the state ordered as `[x, ẋ, θ, θ̇]`, this gives the matrices:

In [ ]:
# physical parameters (match Day_1/cart_pole_dynamics.py)
M, m, L, g = 1.0, 0.3, 0.8, 9.81

A = np.array([
    [0, 1,            0,                0],
    [0, 0,         -m*g/M,              0],
    [0, 0,            0,                1],
    [0, 0,  (M+m)*g/(M*L),               0],
])

B = np.array([
    [0],
    [1/M],
    [0],
    [-1/(M*L)],
])

C = np.eye(4)         # we measure all four states
D = np.zeros((4, 1))

sys = control.ss(A, B, C, D)
print('open-loop poles:', control.poles(sys))

**You should see a pole with positive real part** (something like `+5.5`) — the open-loop pendulum is unstable, as expected. Now design the controller.

In [ ]:
# Q: penalise theta heavily (we care about not falling over),
#    cart position less so, velocities mildly.
Q = np.diag([1.0, 1.0, 50.0, 1.0])
R = np.array([[0.1]])           # accept relatively large forces

K, S, E = control.lqr(sys, Q, R)
print('K (gain row):', K)
print('closed-loop eigenvalues:', E)

A_cl = A - B @ K
assert (np.real(np.linalg.eigvals(A_cl)) < 0).all()
print('all closed-loop poles in left half-plane — controller is stabilising.')

In [ ]:
# Simulate the closed-loop response from a non-zero initial pole angle.
sys_cl = control.ss(A_cl, B, C, D)

x0 = np.array([0, 0, np.radians(5), 0])     # 5 degrees off vertical
t_resp, y_resp = control.initial_response(sys_cl, T=np.linspace(0, 3, 300), X0=x0)

fig, axes = plt.subplots(2, 1, figsize=(8, 4), sharex=True)
axes[0].plot(t_resp, y_resp[0]);                 axes[0].set_ylabel('cart x (m)')
axes[1].plot(t_resp, np.degrees(y_resp[2]));     axes[1].set_ylabel('theta (deg)')
axes[1].set_xlabel('t (s)')
for ax in axes: ax.grid(alpha=0.3)
plt.suptitle('LQR-controlled cart-pole — recovery from 5-degree tilt')
plt.tight_layout(); plt.show()

**What you should see:** `θ` swings back to 0 within a second or two; the cart drifts a bit to apply the corrective force and then settles. That's a working balancing controller.

## Functions used in this project — quick reference

| Function | Signature | What it does |
|----------|-----------|--------------|
| `control.ss`               | `control.ss(A, B, C, D)`              | Build a continuous-time state-space system. |
| `sys.A`, `sys.B`, `sys.C`, `sys.D` | attributes                    | Access the matrices. |
| `control.lqr`              | `K, S, E = control.lqr(sys, Q, R)`    | Optimal LQR gain. |
| `control.poles`            | `control.poles(sys)`                  | Eigenvalues of `A` — stability check. |
| `control.step_response`    | `t, y = control.step_response(sys)`   | Step-input simulation. |
| `control.initial_response` | `t, y = control.initial_response(sys, T=, X0=)` | Free response from a non-zero initial state. |
| `control.bode_plot`        | `control.bode_plot(sys)`              | Frequency-response plot. |

**Workflow:**

1. Linearize the nonlinear dynamics to get `A, B, C, D`.
2. Build `sys = control.ss(A, B, C, D)`.
3. Choose `Q` (state weights) and `R` (effort weight).
4. `K, _, _ = control.lqr(sys, Q, R)`.
5. Verify closed-loop poles `np.linalg.eigvals(A - B @ K)` are all negative real part.
6. Inside your simulation loop, compute `F = -K @ state` and feed it to the (nonlinear) integrator.